# VAST-LoRA: Qwen2.5-3B scale test on Kaggle T4x2

Notebook này clone một commit cố định, chạy cùng một asynchronous trace cho `freshness`, `vast`, `mtip` và `mtip_adaptive`, sau đó ghi bảng kết quả và verdict trực tiếp bên dưới.

**Kaggle settings:** bật Internet và chọn accelerator **GPU T4 x2**. `pilot` chạy một seed để phát hiện lỗi/scaling signal; chỉ `full` với ba seed mới có thể trả verdict `GO`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/TrgPhan/VASTLoRA.git"
REPO_REF = "5c44028"
RUN_MODE = "pilot"  # Change to "full" for 3 seeds and the final GO gate.

WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "VASTLoRA-run"
RESULT_DIR = WORK_ROOT / "vastlora-3b-results"
assert RUN_MODE in {"pilot", "full"}
print({"repo_ref": REPO_REF, "run_mode": RUN_MODE})

## 1. Clone and install the pinned implementation

In [ ]:
if REPO_DIR.exists():
    assert REPO_DIR.parent == WORK_ROOT
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[scale]"],
    check=True,
)
resolved_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
assert resolved_commit.startswith(REPO_REF)
print("Installed commit:", resolved_commit)

## 2. Verify T4x2 and prefetch shared assets

In [ ]:
import torch

subprocess.run(["nvidia-smi"], check=True)
gpu_count = torch.cuda.device_count()
assert gpu_count >= 2, f"Expected Kaggle T4x2, found {gpu_count} CUDA device(s)"
gpu_names = [torch.cuda.get_device_name(index) for index in range(gpu_count)]
print("CUDA devices:", gpu_names)

In [ ]:
from datasets import load_dataset
from huggingface_hub import snapshot_download

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
snapshot_download(MODEL_NAME)
load_dataset("nyu-mll/glue", "sst2")
print("Model and SST-2 are cached for both GPU workers.")

## 3. Validate the asynchronous experiment before loading the model

In [ ]:
RUNNER = REPO_DIR / "scripts/run_kaggle_3b.py"
CONFIG = REPO_DIR / "configs/kaggle_3b_pilot.json"
for method in ("freshness", "vast", "mtip", "mtip_adaptive"):
    subprocess.run(
        [sys.executable, str(RUNNER), "--config", str(CONFIG), "--method", method, "--dry-run"],
        cwd=REPO_DIR,
        check=True,
    )

## 4. Run paired methods on both GPUs

Each process sees exactly one T4. Jobs are launched in pairs so two independent trajectories run concurrently without model sharding. Every method uses the same seed, data partition and async event schedule.

In [ ]:
if RESULT_DIR.exists():
    assert RESULT_DIR.parent == WORK_ROOT
    shutil.rmtree(RESULT_DIR)
RESULT_DIR.mkdir(parents=True)
LOG_DIR = RESULT_DIR / "logs"
LOG_DIR.mkdir()

methods = ["freshness", "vast", "mtip", "mtip_adaptive"]
seeds = [2026] if RUN_MODE == "pilot" else [2026, 2027, 2028]
jobs = [(method, seed) for seed in seeds for method in methods]
extra_args = [] if RUN_MODE == "pilot" else [
    "--collected-returns", "32", "--eval-examples", "256"
]

def launch_job(method, seed, gpu):
    log_path = LOG_DIR / f"{method}_seed{seed}.log"
    log_handle = log_path.open("w", encoding="utf-8")
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": str(gpu),
        "PYTHONUNBUFFERED": "1",
        "TOKENIZERS_PARALLELISM": "false",
    })
    command = [
        sys.executable, str(RUNNER),
        "--config", str(CONFIG),
        "--method", method,
        "--seed", str(seed),
        "--output-dir", str(RESULT_DIR),
        *extra_args,
    ]
    process = subprocess.Popen(
        command, cwd=REPO_DIR, env=env, stdout=log_handle, stderr=subprocess.STDOUT
    )
    return process, log_handle, log_path

started = time.perf_counter()
for wave_start in range(0, len(jobs), 2):
    wave = jobs[wave_start:wave_start + 2]
    running = [launch_job(method, seed, gpu) for gpu, (method, seed) in enumerate(wave)]
    print("Started:", wave)
    while any(process.poll() is None for process, _, _ in running):
        time.sleep(30)
        print("  still running:", [job for job, item in zip(wave, running) if item[0].poll() is None])
    for (method, seed), (process, handle, log_path) in zip(wave, running):
        handle.close()
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]
        print(f"\n--- {method} seed={seed}, exit={process.returncode} ---")
        print("\n".join(tail))
        if process.returncode != 0:
            raise RuntimeError(f"{method} seed={seed} failed; inspect {log_path}")

wall_minutes = (time.perf_counter() - started) / 60
print(f"All {len(jobs)} runs completed in {wall_minutes:.1f} wall-clock minutes.")

## 5. Results written into this notebook

In [ ]:
SUMMARY_SCRIPT = REPO_DIR / "scripts/summarize_kaggle_3b.py"
subprocess.run(
    [sys.executable, str(SUMMARY_SCRIPT), "--input-dir", str(RESULT_DIR)],
    cwd=REPO_DIR,
    check=True,
)

import pandas as pd
from IPython.display import Markdown, display

summary_dir = RESULT_DIR / "summary"
summary = pd.read_csv(summary_dir / "method_summary.csv")
comparisons = pd.read_csv(summary_dir / "paired_comparisons.csv")
verdict = json.loads((summary_dir / "verdict.json").read_text())

display(summary[[
    "method", "seed", "baseline_accuracy", "final_accuracy",
    "final_nll", "accuracy_change_pp", "mean_staleness",
    "mean_rho_after_warmup", "mean_adaptive_left_rank",
    "runtime_seconds", "peak_cuda_memory_gib"
]])
display(comparisons)
display(Markdown((summary_dir / "verdict.md").read_text()))

In [ ]:
import matplotlib.pyplot as plt

mean_metrics = summary.groupby("method", as_index=False).agg(
    final_accuracy=("final_accuracy", "mean"),
    final_nll=("final_nll", "mean"),
    peak_vram=("peak_cuda_memory_gib", "max"),
)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(mean_metrics["method"], 100 * mean_metrics["final_accuracy"])
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Final SST-2 accuracy")
axes[1].bar(mean_metrics["method"], mean_metrics["final_nll"])
axes[1].set_ylabel("NLL (lower is better)")
axes[1].set_title("Final label NLL")
axes[2].bar(mean_metrics["method"], mean_metrics["peak_vram"])
axes[2].set_ylabel("Peak allocated VRAM (GiB)")
axes[2].set_title("Per-process GPU memory")
for axis in axes:
    axis.tick_params(axis="x", rotation=25)
    axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
adaptive_events = []
for seed in seeds:
    path = RESULT_DIR / f"mtip_adaptive_seed{seed}" / "events.csv"
    frame = pd.read_csv(path)
    frame["seed"] = seed
    adaptive_events.append(frame)
adaptive_events = pd.concat(adaptive_events, ignore_index=True)
display(adaptive_events.tail(12))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for seed, frame in adaptive_events.groupby("seed"):
    axes[0].plot(frame["event"], frame["mean_left_rank"], marker="o", label=f"seed {seed}")
    axes[1].plot(frame["event"], frame["rho"], marker="o", label=f"seed {seed}")
axes[0].set(title="Adaptive reference rank", xlabel="Async return", ylabel="Mean left rank")
axes[1].set(title="Retained update energy", xlabel="Async return", ylabel="rho")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
plt.show()

## 6. Reproducibility record and downloadable artifact

In [ ]:
from IPython.display import FileLink

record = {
    "git_commit": resolved_commit,
    "run_mode": RUN_MODE,
    "model": MODEL_NAME,
    "gpus": gpu_names,
    "seeds": seeds,
    "wall_minutes": wall_minutes,
    "verdict": verdict,
}
(RESULT_DIR / "run_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(WORK_ROOT / "vastlora-3b-results"), "zip", RESULT_DIR)
display(record)
display(FileLink(archive))

### Interpretation rule

- `PILOT_GO`: adaptive MTIP beats Freshness by at least `0.5 pp` and improves NLL, but only one seed was run.
- `GO`: the same gate passes over at least three seeds and adaptive MTIP wins accuracy on every seed.
- `INCONCLUSIVE`: accuracy and NLL disagree or the margin is too small.
- `NO_GO`: adaptive MTIP loses at least `0.5 pp` and also worsens NLL.

The notebook deliberately does not contain fabricated pre-run metrics. Saving a Kaggle version after execution preserves every table, plot, log tail and verdict inside the executed notebook.